# Used Car Prices - XGBoost

Predict the selling price (INR) of a used car.

> If you see *No module named ...* or *file not found*, open the project folder itself in Jupyter/VS Code (or run `import os; os.chdir('/path/to/used-car-price-prediction')`).

In [ ]:
%pip install -q xgboost

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

from CarFeatures import add_features, clean_data, encode
from helper_utils import plot_actual_vs_predicted, plot_importance, show_results

## 1. Load the data

Change the file name or path if your CSV is somewhere else.

In [ ]:
df = pd.read_csv("Car details v3.csv")
df.head()

## 2. Clean, add features, encode

Prices are skewed, so the model learns `log(1 + price)`. We convert predictions back to rupees with `np.expm1`.

In [ ]:
df = clean_data(df)
df = add_features(df)
data = encode(df)

X = data.drop(columns='selling_price')
y = np.log1p(data['selling_price'])
print(X.shape)

## 3. Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
y_test_price = np.expm1(y_test)          # real prices, for reporting

## 4. Baseline: Ridge regression

In [ ]:
ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
ridge.fit(X_train, y_train)
show_results(y_test_price, np.expm1(ridge.predict(X_test)), 'Ridge')

## 5. XGBoost (small grid search)

XGBoost builds trees one after another, each fixing the mistakes of the previous ones.

In [ ]:
grid = {'n_estimators': [300, 600], 'max_depth': [4, 6], 'learning_rate': [0.05, 0.1]}
search = GridSearchCV(
    XGBRegressor(random_state=42, n_jobs=-1),
    grid, cv=5, scoring='neg_root_mean_squared_error')
search.fit(X_train, y_train)

print('Best settings:', search.best_params_)
xgb = search.best_estimator_

## 6. Results on the test set

In [ ]:
pred = np.expm1(xgb.predict(X_test))
show_results(y_test_price, pred, 'XGBoost')
plot_actual_vs_predicted(y_test_price, pred)

## 7. Most important features

In [ ]:
plot_importance(xgb, X.columns)

## 8. Conclusions

Write your own: how much better is XGBoost than Ridge? Which features matter most?